In [3]:
import numpy as np
import pandas as pd
from models.sbts_multi import simulateSB_multi
from models.sbts_multi_markov import simulateSB_multi_mark
from metrics.eval_functions import get_stats, get_scores
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from metrics.eval_functions import get_stats, plot_sample_multi
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from metrics.eval_functions import get_stats, get_scores, plot_sample_multi
from metrics.correlation_metrics import correlation_report


In [4]:
# Load data
print("Loading data...")
data = pd.read_csv("data/sp500_top10_prices.csv")
numeric_cols = data.select_dtypes(include=[np.number]).columns
data = data[numeric_cols].ffill().bfill()
asset_names = numeric_cols.tolist()
    
log_returns = np.log(data / data.shift(1)).dropna().values
    
window_size = 10
X_real = np.array([
    log_returns[i:i+window_size] 
    for i in range(len(log_returns) - window_size)
])
    
print(f"✓ Data: {X_real.shape}\n")

Loading data...
✓ Data: (3258, 10, 9)



In [5]:
# Calculer autocorrélation pour chaque asset
print("Autocorrelation (lag=1) for each asset:\n")

for i, col in enumerate(data.columns):
    # ACF simple: corr(t, t-1)
    acf_val = np.corrcoef(log_returns[:-1, i], log_returns[1:, i])[0, 1]
    print(f"{col}: ACF(1) = {acf_val:.6f}")


Autocorrelation (lag=1) for each asset:

AAPL: ACF(1) = -0.040060
MSFT: ACF(1) = -0.019710
NVDA: ACF(1) = -0.105793
TSLA: ACF(1) = -0.042265
AMZN: ACF(1) = -0.075124
GOOGL: ACF(1) = -0.032152
META: ACF(1) = -0.113857
BRK-B: ACF(1) = -0.072007
JNJ: ACF(1) = -0.007853


In [7]:

# ========== LOAD SYNTHETICS ==========
print("Loading synthetic data...")

methods = {}

# SBTS
try:
    X_sbts = np.load('X_synth_returns.npy')
    methods['SBTS'] = X_sbts
    print(f"✓ SBTS (K=0): {X_sbts.shape}")
except FileNotFoundError:
    print("⚠️  SBTS file not found")

# Flow Matching
try:
    X_fm = np.load('X_synth_fm_returns.npy')
    methods['FM'] = X_fm
    print(f"✓ Flow Matching: {X_fm.shape}")
except FileNotFoundError:
    print("⚠️  Flow Matching file not found")

# ICNN
try:
    X_icnn = np.load('X_synth_returns_icnn.npy')
    methods['ICNN'] = X_icnn
    print(f"✓ ICNN: {X_icnn.shape}")
except FileNotFoundError:
    print("⚠️  ICNN file not found")

print(f"\n→ {len(methods)} methods to compare: {list(methods.keys())}\n")

if len(methods) == 0:
    raise RuntimeError("No synthetic data found, aborting.")

# ========== STATISTICAL EVALUATION ==========
print("="*70)
print("STATISTICAL COMPARISON")
print("="*70 + "\n")

stats_dict = {}
for name, X_synth in methods.items():
    print(f"\n{name} STATISTICS:")
    stats = get_stats(X_real, X_synth, col=asset_names)
    print(stats)
    stats_dict[name] = stats

# ========== DISCRIMINATIVE & PREDICTIVE SCORES ==========
print("\n" + "="*70)
print("DISCRIMINATIVE & PREDICTIVE SCORES")
print("="*70 + "\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scores_dict = {}

for name, X_synth in methods.items():
    print(f"Computing {name} scores...")
    try:
        disc, pred = get_scores(
            X_real,
            X_synth,
            col_pred=None,
            itt=1000,
            n_temp=5,
            min_max=False,
            device=device,
        )
        scores_dict[name] = {'disc': disc, 'pred': pred}
        print(f"✓ {name}:")
        print(f"  Discriminative: {disc.mean():.4f} ± {disc.std():.4f}")
        print(f"  Predictive:     {pred.mean():.4f} ± {pred.std():.4f}\n")
    except Exception as e:
        print(f"⚠️  {name} scores failed: {e}\n")
        scores_dict[name] = None

# ========== CORRELATION METRICS ==========
print("="*70)
print("CORRELATION METRICS")
print("="*70 + "\n")

corr_dict = {}
for name, X_synth in methods.items():
    print(f"\n--- {name} ---")
    try:
        corr = correlation_report(X_real, X_synth, asset_names=asset_names)
        corr_dict[name] = corr
    except Exception as e:
        print(f"⚠️  {name} correlation report failed: {e}")
        corr_dict[name] = None

# ========== VISUALIZATION ==========
print("\n" + "="*70)
print("VISUALIZATION")
print("="*70 + "\n")

for name, X_synth in methods.items():
    print(f"Plotting REAL DATA vs {name}...")
    try:
        plot_sample_multi(X_real, X_synth, col=asset_names, x0=0)
        filename = f'{name.lower()}_sample.png'
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Saved: {filename}")
    except Exception as e:
        print(f"⚠️  Plot for {name} failed: {e}")

# ========== FINAL RECAP TABLE ==========
print("\n" + "="*70)
print("FINAL RECAP")
print("="*70 + "\n")

# Header
header = f"{'Metric':<25}"
for name in methods.keys():
    header += f"{name:<15}"
print(header)
print("-" * len(header))

# Discriminative scores
row = f"{'Discriminative':<25}"
for name in methods.keys():
    if scores_dict.get(name) is not None:
        d = scores_dict[name]['disc']
        row += f"{d.mean():.4f}±{d.std():.3f}".ljust(15)
    else:
        row += "N/A".ljust(15)
print(row)

# Predictive scores
row = f"{'Predictive':<25}"
for name in methods.keys():
    if scores_dict.get(name) is not None:
        p = scores_dict[name]['pred']
        row += f"{p.mean():.4f}±{p.std():.3f}".ljust(15)
    else:
        row += "N/A".ljust(15)
print(row)

# Correlation metrics
for metric_key, metric_label in [
    ('auto_corr', 'Auto-corr (raw)'),
    ('abs_corr', 'Auto-corr (abs/vol)'),
    ('cross_corr', 'Cross-corr'),
]:
    row = f"{metric_label:<25}"
    for name in methods.keys():
        if corr_dict.get(name) is not None:
            row += f"{corr_dict[name][metric_key]:.4f}".ljust(15)
        else:
            row += "N/A".ljust(15)
    print(row)


Loading synthetic data...
✓ SBTS (K=0): (1000, 10, 9)
✓ Flow Matching: (1000, 10, 9)
✓ ICNN: (1000, 10, 9)

→ 3 methods to compare: ['SBTS', 'FM', 'ICNN']

STATISTICAL COMPARISON


SBTS STATISTICS:
         1% Data  1% SBTS  99% Data  99% SBTS  Mean Data  Mean SBTS  Std Data  \
Feature                                                                         
AAPL      -0.049   -0.050     0.047     0.048      0.001      0.001     0.018   
MSFT      -0.058   -0.056     0.056     0.053      0.001      0.001     0.020   
NVDA      -0.030   -0.032     0.031     0.029      0.001      0.000     0.012   
TSLA      -0.048   -0.049     0.048     0.044      0.001      0.001     0.018   
AMZN      -0.029   -0.031     0.028     0.028      0.000      0.000     0.011   
GOOGL     -0.060   -0.061     0.060     0.058      0.001      0.001     0.024   
META      -0.043   -0.044     0.045     0.043      0.001      0.001     0.017   
BRK-B     -0.075   -0.072     0.073     0.072      0.002      0.002     0

/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ori_data = torch.tensor(ori_data, dtype=torch.float32).to(device)
/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  generated_data = torch.tensor(generated_data, dtype=torch.float32).to(device)


Expected finish time: 21:01:39


/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ori_data = torch.tensor(ori_data, dtype=torch.float32).to(device)
/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  generated_data = torch.tensor(generated_data, dtype=torch.float32).to(device)
/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_

Discriminative score (lower the better): 0.019 +- 0.007
Predictive score (lower the better): 0.025 +- 0.0
✓ SBTS:
  Discriminative: 0.0185 ± 0.0066
  Predictive:     0.0254 ± 0.0001

Computing FM scores...
Start time: 21:01:45


...

Discriminative score (lower the better): 0.023 +- 0.007
Predictive score (lower the better): 0.025 +- 0.0
✓ FM:
  Discriminative: 0.0230 ± 0.0071
  Predictive:     0.0254 ± 0.0004

Computing ICNN scores...
Start time: 21:02:21


/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ori_data = torch.tensor(ori_data, dtype=torch.float32).to(device)
/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  generated_data = torch.tensor(generated_data, dtype=torch.float32).to(device)


Expected finish time: 21:02:50


/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ori_data = torch.tensor(ori_data, dtype=torch.float32).to(device)
/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  generated_data = torch.tensor(generated_data, dtype=torch.float32).to(device)
/Users/taniaadmane/Desktop/S2 ENSAE/optimal transport/Optimal-Transport/metrics/predictive_score.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_

Discriminative score (lower the better): 0.098 +- 0.037
Predictive score (lower the better): 0.026 +- 0.0
✓ ICNN:
  Discriminative: 0.0980 ± 0.0369
  Predictive:     0.0261 ± 0.0002

CORRELATION METRICS


--- SBTS ---
CORRELATION METRICS (lower = better fit)

Auto-correlation score (raw returns, lags 1-5):
  Mean: 0.0079
  AAPL: 0.0083
  MSFT: 0.0071
  NVDA: 0.0105
  TSLA: 0.0042
  AMZN: 0.0060
  GOOGL: 0.0073
  META: 0.0079
  BRK-B: 0.0102
  JNJ: 0.0096

Abs-returns ACF score (volatility clustering, lags 1-5):
  Mean: 0.0052
  AAPL: 0.0048
  MSFT: 0.0044
  NVDA: 0.0068
  TSLA: 0.0055
  AMZN: 0.0054
  GOOGL: 0.0060
  META: 0.0065
  BRK-B: 0.0030
  JNJ: 0.0040

Cross-correlation score (off-diagonal Frobenius RMSE):
  Score: 0.0211

--- FM ---
CORRELATION METRICS (lower = better fit)

Auto-correlation score (raw returns, lags 1-5):
  Mean: 0.0071
  AAPL: 0.0041
  MSFT: 0.0084
  NVDA: 0.0069
  TSLA: 0.0045
  AMZN: 0.0074
  GOOGL: 0.0085
  META: 0.0094
  BRK-B: 0.0087
  JNJ: 0.0062

Abs-re